# MASLD-LLM · 07 — v6 trained-on-C3 training + eval

One-cell driver for `run_v6.py` (already on Drive at masld-llm/run_v6.py).

v6 = the v5 recipe plus two new TRAIN corpora in the mixture:
- MB-12xx staged liver (GSE135251 F0-F4 + NAS, GSE167523 NAFL/NASH)
- MB-88xx DRUG-seq (HepG2, 40 compounds)

After training 0.6B and 1.7B LoRA adapters it evaluates the FROZEN tests
(MB-20xx, 85xx, 87xx, 88xx, 86xx) — the trained-on C3 readout.

Runtime: T4 ~12-16 h total, A100/L4 faster. Hourly checkpoints to Drive;
a disconnect resumes from the last checkpoint (script re-run safe).

In [1]:
# Cell 1: mount + run everything (resume-safe)
from google.colab import drive
drive.mount('/content/drive')
!python /content/drive/MyDrive/masld-llm/run_v6.py

Mounted at /content/drive
workspace OK; corpora present
>>> cd /content && (git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git 2>/dev/null || true)
>>> cd /content/LLaMA-Factory && pip -q install -e '.[torch,metrics]' bitsandbytes peft transformers accelerate liger-kernel
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 900.2/900.2 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 42.

In [2]:
# Cell 2: verify outputs landed in Drive
import os, glob
R = '/content/drive/MyDrive/masld-llm/results'
print('checkpoints:', [os.path.basename(p) for p in glob.glob('/content/drive/MyDrive/masld-llm/checkpoints/*v6*')])
print('archives:', [os.path.basename(p) for p in glob.glob(R + '/frontier_runs/*v6*')])
print('manifest:', os.path.exists(R + '/v6_run_manifest.json'))

checkpoints: ['qwen3-0.6b-masld-sft-v6', 'qwen3-1.7b-masld-sft-v6']
archives: ['qwen3-0.6b-masld-v6__MB-20xx.jsonl.gz', 'qwen3-0.6b-masld-v6__MB-85xx.jsonl.gz', 'qwen3-0.6b-masld-v6__MB-87xx.jsonl.gz', 'qwen3-0.6b-masld-v6__MB-88xx.jsonl.gz', 'qwen3-1.7b-masld-v6__MB-20xx.jsonl.gz', 'qwen3-1.7b-masld-v6__MB-85xx.jsonl.gz', 'qwen3-1.7b-masld-v6__MB-87xx.jsonl.gz', 'qwen3-1.7b-masld-v6__MB-88xx.jsonl.gz', 'qwen3-0.6b-masld-v6__MB-86xx.jsonl.gz', 'qwen3-1.7b-masld-v6__MB-86xx.jsonl.gz']
manifest: True
